# D2.3 · Scoping an agentic incident

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.2 · When the actor is an agent](https://spbreed.github.io/cyber-commons/lessons/D2.2.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Scope a multi-agent incident end to end.

**Why a security engineer needs it.** The initiating agent is not the acting one. The control it builds is: reconstruct the action chain across all three planes.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The agent acted for eleven minutes on delegated credentials at machine speed. Scoping that means reconstructing blast radius from identity and egress logs, because asking what it touched is not a question anyone can answer from memory.

> **At CyberTravels.** Eleven minutes of CyberTravels on delegated credentials. What it touched is not answerable from memory — it comes out of the identity and egress logs, if they exist. R9.

## 2 · The framework

```
   11 minutes at machine speed

   identity log ---+                    +--> resources touched
                   +--> reconstruct --> +--> data read
   egress log   ---+                    +--> destinations reached
                                        +--> credentials used

   the question "what did it touch" is not answerable from memory
```

Scoping answers "what was touched?" For a host-based incident you enumerate
hosts. For an agentic incident, **scope follows the delegation graph**.

The agent that touched the resource is usually the *last* actor in a chain. If
you scope only that actor, you miss everything the earlier actors reached — and
because authority narrows down the chain, the earlier actors typically had
*more* access, not less.

The undercount is systematic and it grows with delegation depth, which is the
operational reason B2.0 bounds delegation depth in the first place.

## 3 · Scoping as a skill

Scoping a human incident asks where someone logged in. Scoping this one asks what the agent **decided** — every action was individually authorised, so nothing looks wrong at the authentication layer.

Two fields in the contract carry most of the weight. `reach` and `confirmed_exfiltration` are separate numbers, because reach is the scope until proven otherwise and the smaller number must never stand in for the larger in a notification decision. And `does_not_stop` makes containment state its own limits.

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/secops/incident-scoping/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: incident-scoping
description: >-
  Scope an incident in which an agent was the actor — what it touched, what it
  changed, what it may have exfiltrated, and where containment must cut. Use
  when responding to an agent-involved incident, reconstructing what an
  autonomous system did, deciding what to revoke, or sizing notification
  obligations.
allowed-tools: Read, Grep, Bash
---

# Scoping an agentic incident

Scoping a human incident asks where someone logged in. Scoping an agentic one
asks what the agent **decided**, because a compromised agent's actions are all
individually authorised. Nothing looks anomalous at the authentication layer;
the anomaly is in the sequence.

## When to use this

Any incident where an agent, an automated pipeline, or an AI-driven tool
performed actions under investigation.

## Procedure

**1 — Fix the window.** Establish the first suspicious decision, not the first
alert. Work backwards from the earliest action you cannot explain; the trigger
is usually earlier than the detection by the length of one task loop.

**2 — Reconstruct the decision chain.** For the window, list every action with
the input that motivated it. The critical question is which input entered the
context from **outside the trust boundary** — a fetched page, an issue comment,
a dependency's README, a tool description. That input is the likely root cause,
and it is invisible if you only log tool calls and not their justification.

**3 — Separate authority from behaviour.** For each action ask: was it within
the agent's granted authority? Actions that were authorised but wrong tell you
the grant was too broad. Actions that exceeded authority tell you a control
failed. These lead to different fixes and must not be pooled.

**4 — Establish data reach.** What did the agent read, and where could it have
sent it? Reach is bounded by the agent's egress, not by what it appears to have
sent — a request body you cannot see is still reach. State reach and confirmed
exfiltration as separate numbers, and never let the smaller one stand in for
the larger in a notification decision.

**5 — Decide the containment cut.** Options, in increasing cost: revoke the
agent's credential, disable the trigger, quarantine the workload, disable the
whole class of agents. Choose by blast radius, not by convenience, and record
what the cut does **not** stop — sibling agents on the same shared service
account almost always survive a credential revocation aimed at one of them.

**6 — Preserve evidence the agent could alter.** If the agent can write to the
log store, the logs are not evidence. Snapshot first, then contain.

## Output contract

```json
{
  "window": {"first_suspicious_action": "str", "detected_at": "str", "gap_seconds": 0},
  "chain": [{"action": "str", "motivating_input": "str",
             "input_origin": "operator|internal|external_untrusted",
             "within_authority": true}],
  "root_cause": {"input": "str", "origin": "str", "why_trusted": "str"},
  "authority": {"authorised_but_wrong": 0, "exceeded_authority": 0},
  "data": {"reach": ["str"], "confirmed_exfiltration": ["str"], "egress_bounded_by": "str"},
  "containment": {"cut": "credential|trigger|workload|class",
                  "does_not_stop": ["str"], "evidence_snapshotted_first": true},
  "clock": {"regulatory_trigger": false, "basis": "str"}
}
```

## Failure modes

- **Scoping by authentication.** Every action was authenticated; that is the
  point.
- **Logging tool calls without their motivating input.** Root cause then cannot
  be established at all.
- **Reporting confirmed exfiltration as the scope.** Reach is the scope until
  proven otherwise.
- **Revoking one agent's token** when the identity is shared, and calling it
  contained.
- **Containing before snapshotting** a log store the agent can write to.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## What you just proved

Scoping the last actor finds `cluster-prod` alone; the whole chain reaches six resources, missing five, with an undercount factor of 6.0. The undercount grows with each hop. Transitive scoping adds five second-order identities that shared a resource, explicitly marked as in scope rather than confirmed compromised.

## Your turn

For your last incident involving a service account, recompute the scope by walking what else that account could reach. The number is almost always larger than what was written in the report.

---

**Next → [D2.4 · Containment at machine speed](https://spbreed.github.io/cyber-commons/lessons/D2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*